# Case study: the cleaned review table with a log of cleaning decisions

*Session 4, block 3 · Author: course team · Licence: CC-BY-4.0*

**Task.** Produce the cleaned review table that later sessions use, and a log that records each cleaning
decision: the rule, how many rows it affected, and why.

The pipeline is a sequence of small functions. Each function takes a DataFrame, returns a new one and writes
an entry into the log. Nothing is changed by hand.

Output: `case-study/data/reviews_clean.parquet` and `cleaning_log.csv` next to it (the `data` folder is not
part of the repository). Set the environment variable `CLEAN_OUT` to write somewhere else.

In [ ]:
from pathlib import Path

# find the repository root (the folder that contains case-study/), wherever the notebook runs
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").is_dir())
DATA = ROOT / "case-study" / "data"
assert (DATA / "train.parquet").exists(), "run case-study/prepare_data.py first"

import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

OUT = Path(os.environ.get("CLEAN_OUT", DATA / "reviews_clean.parquet"))
raw = pd.read_parquet(DATA / "train.parquet")
products = pd.read_parquet(DATA / "products.parquet")
print(raw.shape)

## 1. The decision log

In [ ]:
log = []


def record(step: str, rule: str, affected: int, action: str, reason: str, n_rows: int) -> None:
    log.append({"step": step, "rule": rule, "rows_affected": affected, "action": action,
                "reason": reason, "rows_after": n_rows})

## 2. Cleaning steps

Each step follows the same pattern: find the rows, decide (remove, correct, flag, transform), record.

In [ ]:
def drop_empty_text(df):
    empty = df["text"].str.strip().eq("")
    out = df[~empty].copy()
    record("1 completeness", "text empty or white space", int(empty.sum()), "remove",
           "no content for text analysis; rating alone is kept in the raw data", len(out))
    return out


def normalise_text(df):
    out = df.copy()
    has_br = out["text"].str.contains("<br", regex=False)
    has_nbsp = out["text"].str.contains("\u00a0", regex=False)
    for col in ["title", "text"]:
        out[col] = (out[col].str.replace(r"<br\s*/?>", " ", regex=True)    # HTML line breaks
                            .str.replace("\u00a0", " ", regex=False)        # no-break space
                            .str.replace(r"\s+", " ", regex=True)           # repeated white space
                            .str.strip())
    record("2 validity", "HTML <br /> tags or U+00A0 in text", int((has_br | has_nbsp).sum()), "correct",
           "markup and special spaces distort lengths and word counts", len(out))
    return out


def dedupe_user_product(df):
    dup = df.duplicated(["user_id", "parent_asin"], keep="last")
    out = df.sort_values("date").drop_duplicates(["user_id", "parent_asin"], keep="last")
    record("3 uniqueness", "more than one review by the same user for the same product", int(dup.sum()),
           "remove all but the latest", "an updated review replaces the earlier opinion; avoids counting one customer twice", len(out))
    return out.sort_index()


def flag_duplicate_texts(df):
    out = df.copy()
    out["is_duplicate_text"] = out["text"].str.lower().duplicated(keep=False)
    record("4 uniqueness", "same text as another review (e.g. 'Great product')", int(out["is_duplicate_text"].sum()),
           "flag (keep)", "short generic texts are genuine; removing them would bias towards long reviews", len(out))
    return out

In [ ]:
df = raw.pipe(drop_empty_text).pipe(normalise_text).pipe(dedupe_user_product).pipe(flag_duplicate_texts)
pd.DataFrame(log)

## 3. Derived columns and transformations

Text length and helpful votes are strongly right-skewed. We keep the raw columns and add transformed ones.
A **Box–Cox** transformation estimates a power λ that makes the distribution as close to normal as possible
(λ = 0 is the logarithm); it requires positive values. **Yeo–Johnson** also allows 0 and negative values,
which suits `helpful_vote`.

In [ ]:
from scipy.stats import skew
from sklearn.preprocessing import PowerTransformer

df["year"] = df["date"].dt.year
df["text_len"] = df["text"].str.len()
df["n_words"] = df["text"].str.split().str.len()

bc = PowerTransformer(method="box-cox", standardize=False).fit(df[["text_len"]])
df["text_len_boxcox"] = bc.transform(df[["text_len"]]).ravel()
yj = PowerTransformer(method="yeo-johnson", standardize=False).fit(df[["helpful_vote"]])
df["helpful_vote_yj"] = yj.transform(df[["helpful_vote"]]).ravel()
df["log1p_helpful_vote"] = np.log1p(df["helpful_vote"])

print(f"Box-Cox lambda for text_len: {bc.lambdas_[0]:.3f}; Yeo-Johnson lambda for helpful_vote: {yj.lambdas_[0]:.3f}")
pd.DataFrame({
    "text_len": [skew(df["text_len"]), skew(np.log(df["text_len"])), skew(df["text_len_boxcox"])],
    "helpful_vote": [skew(df["helpful_vote"]), skew(df["log1p_helpful_vote"]), skew(df["helpful_vote_yj"])],
}, index=["raw", "log / log1p", "Box-Cox / Yeo-Johnson"]).round(2)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3), layout="constrained")
for ax, col, title in zip(axes, ["text_len", "text_len_boxcox", "log1p_helpful_vote"],
                          ["text length (characters)", "text length, Box-Cox", "log(1 + helpful votes)"]):
    ax.hist(df[col], bins=60, color="#2a78d6")
    ax.set(title=title)
    ax.spines[["top", "right"]].set_visible(False)
plt.show()
record("5 transformation", "text_len, helpful_vote strongly right-skewed", len(df), "add transformed columns",
       f"Box-Cox (lambda = {bc.lambdas_[0]:.2f}) and Yeo-Johnson / log1p; raw columns kept", len(df))

## 4. Outliers: flag, do not delete

**Univariate.** The MAD rule on the Box–Cox scale flags unusually short or long reviews.

**Multivariate.** The Mahalanobis distance on (log text length, log title length) flags unusual
*combinations*, such as a very long title with a one-word text. We estimate centre and covariance robustly
with the minimum covariance determinant (MCD) on a random sample of 20,000 reviews and compare the squared
distance with the 99.9 % quantile of the chi-square distribution with 2 degrees of freedom.

In [ ]:
from scipy.stats import chi2
from sklearn.covariance import MinCovDet

x = df["text_len_boxcox"]
mad = (x - x.median()).abs().median()
df["outlier_length_mad"] = (0.6745 * (x - x.median()) / mad).abs() > 3.5
record("6 outliers (univariate)", "robust z-score of Box-Cox text length > 3.5", int(df["outlier_length_mad"].sum()),
       "flag (keep)", "very short or very long reviews are genuine; analyses can exclude them", len(df))

Z = np.column_stack([np.log1p(df["text_len"]), np.log1p(df["title"].str.len())])
sample = Z[np.random.default_rng(2026).choice(len(Z), 20_000, replace=False)]
mcd = MinCovDet(random_state=0).fit(sample)
df["mahalanobis_d2"] = mcd.mahalanobis(Z)
df["outlier_mahalanobis"] = df["mahalanobis_d2"] > chi2.ppf(0.999, df=2)
record("7 outliers (multivariate)", "robust Mahalanobis d2 of (log text length, log title length) > chi2(2) 99.9 %",
       int(df["outlier_mahalanobis"].sum()), "flag (keep)", "unusual title/text combinations; inspect before use", len(df))
df.loc[df["outlier_mahalanobis"], ["title", "text", "text_len"]].sample(5, random_state=1)

## 5. Product attributes

We add the product columns that later sessions use. A missing price stays missing, with an indicator
(block 2: imputation adds little information here). A missing store becomes the category `"unknown"`.
`categories` is empty for every product and is not carried over.

In [ ]:
prod = products[["parent_asin", "store", "price"]]
df = df.merge(prod, on="parent_asin", how="left", validate="many_to_one")
n_store_missing = int(df["store"].isna().sum())
df["store"] = df["store"].fillna("unknown")
df["price_missing"] = df["price"].isna()
record("8 completeness", "store of the product missing", n_store_missing, "fill with 'unknown'",
       "keeps the products in group comparisons; 'unknown' is its own group", len(df))
record("9 completeness", "price missing", int(df["price_missing"].sum()), "keep NULL, add indicator price_missing",
       "missingness depends on popularity (not MCAR); imputation recovers little (workbook 08)", len(df))
record("10 completeness", "products.categories empty for all products", len(products), "drop column",
       "no information", len(df))

## 6. Validate the result, then save it

In [ ]:
assert df["review_id"].is_unique
assert not df.duplicated(["user_id", "parent_asin"]).any()
assert df["text"].str.len().gt(0).all()
assert not df["text"].str.contains("<br", regex=False).any()
assert df["rating"].between(1, 5).all()
assert len(df) == len(raw) - log[0]["rows_affected"] - log[2]["rows_affected"]
print("all assertions passed:", df.shape)

In [ ]:
OUT.parent.mkdir(parents=True, exist_ok=True)
df.to_parquet(OUT, index=False)
log_df = pd.DataFrame(log)
log_df.to_csv(OUT.with_name("cleaning_log.csv"), index=False)
print(f"wrote {OUT} ({OUT.stat().st_size / 1e6:.0f} MB) and {OUT.with_name('cleaning_log.csv').name}")
log_df

**TODO.** (1) Add a step of your own, for example removing reviews written in another language, and record
it. (2) Run the tests in [`quality/test_review_quality.py`](quality/test_review_quality.py); the tests on the
cleaned table should now pass. (3) Write three sentences for the data card that summarise the cleaning.